In [5]:
import sounddevice as sd
import queue
import json
import sys
import os
import csv
import wave
from datetime import datetime
from vosk import Model, KaldiRecognizer
from IPython.display import clear_output, display

# --- 1. 設定 ---
CSV_FILE = "data/karuta_data.csv"
RESULTS_CSV = "results/recognition_history.csv" 
SAMPLERATE = 16000

# --- 2. 辞書ロジック ---

def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    if pos_index == 0 or pos_index >= 2:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    return p

def extract_kanji_parts(text):
    text = text.replace(" ", "")
    p = []
    if len(text) >= 2: p.append(text[:2])
    if len(text) >= 3: p.append(text[:3])
    if len(text) >= 2: p.append(text[-2:])
    if len(text) >= 3: p.append(text[-3:])
    return p

def prepare_karuta_system(file_path):
    if not os.path.exists(file_path):
        print(f"Error: {file_path} が見つかりません。")
        return None, None, None
    raw_cards = []
    with open(file_path, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            words = []; seen = set()
            def add_words(new_words, is_kimari=False):
                for w in new_words:
                    if w in seen: continue
                    if len(w) >= (1 if is_kimari else 2):
                        words.append(w); seen.add(w)
            add_words([kimari], is_kimari=True)
            add_words([gendai.replace(" ", "")])
            k_blocks = kanji.split(); h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks): add_words(extract_kanji_parts(k_blocks[i]))
                if i < len(h_blocks): add_words(extract_kana_parts(h_blocks[i], i))
            raw_cards.append({"num": num, "words": words})
    
    if not raw_cards: return None, None, None
    min_count = min(len(c["words"]) for c in raw_cards)
    tag_map = {}; grammar_list = []; card_max_scores = {}
    for c in raw_cards:
        num = c["num"]
        unified_words = c["words"][:min_count]
        total_potential = 0
        for w in unified_words:
            grammar_list.append(w)
            tag_map.setdefault(w, []).append(num)
            total_potential += len(w)
        card_max_scores[num] = total_potential
    return list(set(grammar_list)), tag_map, card_max_scores

# --- 3. 表示と保存の関数 ---

def print_dashboard(scores, max_scores, partial_text, model_name, prev_札, target_札, detail_logs):
    clear_output(wait=True)
    results = []
    for tid, score in scores.items():
        max_s = max_scores.get(tid, 1)
        sim = (score / max_s) * 100
        results.append({"id": tid, "score": score, "max": max_s, "sim": sim})
    top_5 = sorted(results, key=lambda x: x['sim'], reverse=True)[:5]
    output = ["=" * 80, f"設定: [Model: {model_name}] [Prev: {prev_札}] [Target: {target_札}]", 
              "録音中 & リアルタイム類似度判定開始 [停止は中断ボタンをクリック]", "=" * 80]
    clean_partial = partial_text.replace("\n", " ")
    display_partial = clean_partial[-50:] if len(clean_partial) > 50 else clean_partial
    output.append(f" | 認識中の言葉  | {display_partial.ljust(50)}")
    output.extend(["-" * 80, f"{'順位':<4}｜{'推定札':<10} | {'加点/満点':<12}｜{'類似度':<8} ｜棒グラフ", "-" * 80])
    for i in range(5):
        if i < len(top_5):
            res = top_5[i]; bar_count = int(res['sim'] / 5); bar = "■" * bar_count + "□" * (20 - bar_count)
            output.append(f"{i+1}位 ｜ {res['id']:>3}番      | {res['score']:>3} / {res['max']:>3} pts | {res['sim']:>5.1f}%    ｜|{bar}")
        else: output.append(f"{i+1}位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|{'□'*20}")
    
    output.extend(["-" * 80, " 【リアルタイム認識ログ（直近5件）】", "-" * 80])
    if detail_logs:
        recent_logs = detail_logs[-5:][::-1]
        for log in recent_logs:
            output.append(f" [{log[0]}] 単語: {log[1]:<8} ➔ {log[2]:>3}番札に加点 (スコア: {log[3]}/{log[4]}, 類似度: {log[5]})")
    else:
        output.append(" まだキーワードが認識されていません。")
        
    output.extend(["=" * 80])
    print("\n".join(output))

def save_to_result_csv(data):
    os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
    file_exists = os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f) 
        if not file_exists: writer.writerow(["日時", "ファイル名", "前の札", "対象札", "最終推定ID", "最終類似度", "認識単語"])
        writer.writerow(data)

# --- 4. メイン処理 ---

def run_realtime_karuta_with_record():
    print("【使用モデルを選択してください】")
    choice = input("1: Standard (精度重視) / 2: Small (速度重視) -> ")
    model_name = "model_standard" if choice == "1" else "model_small"
    prev_name = input("【前の札名を入力してください】: ")
    target_name = input("【識別したい札名を入力してください】: ")
    
    grammar, tag_map, max_scores = prepare_karuta_system(CSV_FILE)
    if not tag_map: return

    print(f"\n--- モデル読み込み中 ({model_name}) ---")
    model = Model(model_name)
    rec = KaldiRecognizer(model, SAMPLERATE, json.dumps(grammar, ensure_ascii=False))

    dt = datetime.now(); dt_str = dt.strftime("%Y%m%d_%H%M%S")
    base_name = f"{prev_name}_{target_name}_{dt_str}"
    save_filename = f"{base_name}.wav"
    detail_log_filename = f"{base_name}.csv" 
    
    # フォルダの作成を明示的に分ける
    os.makedirs("recordings", exist_ok=True)
    os.makedirs("results", exist_ok=True) # resultsフォルダも作成

    # ★詳細ログCSVの保存先を「results」フォルダに変更
    full_path_wav = os.path.join("recordings", save_filename)
    full_path_csv = os.path.join("results", detail_log_filename) 

    q = queue.Queue(); recorded_frames = [] 
    def callback(indata, frames, time, status):
        data = bytes(indata); q.put(data); recorded_frames.append(data)

    scores = {}; seen_words_per_card = {}; all_detected_words = []
    detail_logs = []

    try:
        with sd.RawInputStream(samplerate=SAMPLERATE, blocksize=4000, 
                               channels=1, dtype='int16', callback=callback):
            while True:
                data = q.get()
                if not rec.AcceptWaveform(data):
                    partial = json.loads(rec.PartialResult()).get('partial', "")
                    if partial:
                        new_words = partial.split()
                        for word in new_words:
                            if word in tag_map:
                                for tid in tag_map[word]:
                                    if tid not in seen_words_per_card: seen_words_per_card[tid] = set()
                                    if word not in seen_words_per_card[tid]:
                                        scores[tid] = scores.get(tid, 0) + len(word)
                                        seen_words_per_card[tid].add(word)
                                        if word not in all_detected_words: all_detected_words.append(word)
                                        
                                        current_sim = (scores[tid] / max_scores.get(tid, 1)) * 100
                                        detail_logs.append([
                                            datetime.now().strftime("%H:%M:%S.%f")[:-3],
                                            word, tid, scores[tid], max_scores.get(tid), f"{current_sim:.1f}%"
                                        ])
                    
                    print_dashboard(scores, max_scores, partial, model_name, prev_name, target_name, detail_logs)

    except KeyboardInterrupt:
        final_id, final_sim = "---", 0.0
        if scores:
            final_id = max(scores, key=lambda k: scores[k]/max_scores.get(k, 1))
            final_sim = (scores[final_id] / max_scores.get(final_id, 1)) * 100

        # WAV保存（recordings/配下）
        with wave.open(full_path_wav, 'wb') as wf:
            wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(SAMPLERATE)
            wf.writeframes(b''.join(recorded_frames))
        
        # 詳細ログCSV保存 (results/配下に保存されるようになりました)
        with open(full_path_csv, "w", encoding="utf-8-sig", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["時刻", "認識単語", "加点札ID", "現在スコア", "満点スコア", "類似度"])
            writer.writerows(detail_logs)

        # 全体履歴CSV保存（results/配下）
        save_to_result_csv([dt.strftime("%Y-%m-%d %H:%M:%S"), save_filename, prev_name, target_name, final_id, f"{final_sim:.1f}%", " ".join(all_detected_words)])
        
        print(f"\n録音保存: {full_path_wav}")
        print(f"詳細ログ保存: {full_path_csv}")
        print(f"最終推定: {final_id} ({final_sim:.1f}%)")

if __name__ == "__main__":
    run_realtime_karuta_with_record()

設定: [Model: model_small] [Prev: nanishi] [Target: harusu]
録音中 & リアルタイム類似度判定開始 [停止は中断ボタンをクリック]
 | 認識中の言葉  | くる ある すぎ 長月 にい けら 白妙                              
--------------------------------------------------------------------------------
順位  ｜推定札        | 加点/満点       ｜類似度      ｜棒グラフ
--------------------------------------------------------------------------------
1位 ｜   2番      |   6 /  46 pts |  13.0%    ｜|■■□□□□□□□□□□□□□□□□□□
2位 ｜  37番      |   2 /  45 pts |   4.4%    ｜|□□□□□□□□□□□□□□□□□□□□
3位 ｜  52番      |   2 /  45 pts |   4.4%    ｜|□□□□□□□□□□□□□□□□□□□□
4位 ｜  40番      |   2 /  46 pts |   4.3%    ｜|□□□□□□□□□□□□□□□□□□□□
5位 ｜  35番      |   2 /  46 pts |   4.3%    ｜|□□□□□□□□□□□□□□□□□□□□
--------------------------------------------------------------------------------
 【リアルタイム認識ログ（直近5件）】
--------------------------------------------------------------------------------
 [15:13:52.955] 単語: 白妙       ➔   4番札に加点 (スコア: 2/46, 類似度: 4.3%)
 [15:13:52.955] 単語: 白妙       ➔   2番札に加点 (スコア: 6/46, 類似度: 13

音声検出＆前の札の下の句の除去

In [10]:
import sounddevice as sd
import queue
import json
import sys
import os
import csv
import wave
import time  # 経過時間測定用に追加
from datetime import datetime
from vosk import Model, KaldiRecognizer
from IPython.display import clear_output, display

# --- 1. 設定 ---
CSV_FILE = "data/karuta_data.csv"
RESULTS_CSV = "results/recognition_history.csv" 
SAMPLERATE = 16000

# ★【新規追加】前の札の下の句を無視する時間（秒）
# 小数点第一位までで自由に変更可能です
SKIP_SECONDS = 6.0

# --- 2. 辞書ロジック ---

def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    if pos_index == 0 or pos_index >= 2:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    return p

def extract_kanji_parts(text):
    text = text.replace(" ", "")
    p = []
    if len(text) >= 2: p.append(text[:2])
    if len(text) >= 3: p.append(text[:3])
    if len(text) >= 2: p.append(text[-2:])
    if len(text) >= 3: p.append(text[-3:])
    return p

def prepare_karuta_system(file_path):
    if not os.path.exists(file_path):
        print(f"Error: {file_path} が見つかりません。")
        return None, None, None
    raw_cards = []
    with open(file_path, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            words = []; seen = set()
            def add_words(new_words, is_kimari=False):
                for w in new_words:
                    if w in seen: continue
                    if len(w) >= (1 if is_kimari else 2):
                        words.append(w); seen.add(w)
            add_words([kimari], is_kimari=True)
            add_words([gendai.replace(" ", "")])
            k_blocks = kanji.split(); h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks): add_words(extract_kanji_parts(k_blocks[i]))
                if i < len(h_blocks): add_words(extract_kana_parts(h_blocks[i], i))
            raw_cards.append({"num": num, "words": words})
    
    if not raw_cards: return None, None, None
    min_count = min(len(c["words"]) for c in raw_cards)
    tag_map = {}; grammar_list = []; card_max_scores = {}
    for c in raw_cards:
        num = c["num"]
        unified_words = c["words"][:min_count]
        total_potential = 0
        for w in unified_words:
            grammar_list.append(w)
            tag_map.setdefault(w, []).append(num)
            total_potential += len(w)
        card_max_scores[num] = total_potential
    return list(set(grammar_list)), tag_map, card_max_scores

# --- 3. 表示と保存の関数 ---

# 引数に「is_skipping」と「elapsed」を追加
def print_dashboard(scores, max_scores, partial_text, model_name, prev_札, target_札, detail_logs, is_skipping, elapsed):
    clear_output(wait=True)
    results = []
    for tid, score in scores.items():
        max_s = max_scores.get(tid, 1)
        sim = (score / max_s) * 100
        results.append({"id": tid, "score": score, "max": max_s, "sim": sim})
    top_5 = sorted(results, key=lambda x: x['sim'], reverse=True)[:5]
    
    # 状態表示のラベルを作成
    if is_skipping:
        status_label = f"【下の句スキップ中: {elapsed:.1f}s / {SKIP_SECONDS:.1f}s】(加点無効)"
    else:
        status_label = f"【類似度判定中: {elapsed:.1f}s経過】(加点有効)"

    output = ["=" * 80, 
              f"設定: [Model: {model_name}] [Prev: {prev_札}] [Target: {target_札}]", 
              f"状況: {status_label}", "=" * 80]
              
    clean_partial = partial_text.replace("\n", " ")
    display_partial = clean_partial[-50:] if len(clean_partial) > 50 else clean_partial
    output.append(f" | 認識中の言葉  | {display_partial.ljust(50)}")
    output.extend(["-" * 80, f"{'順位':<4}｜{'推定札':<10} | {'加点/満点':<12}｜{'類似度':<8} ｜棒グラフ", "-" * 80])
    for i in range(5):
        if i < len(top_5):
            res = top_5[i]; bar_count = int(res['sim'] / 5); bar = "■" * bar_count + "□" * (20 - bar_count)
            output.append(f"{i+1}位 ｜ {res['id']:>3}番      | {res['score']:>3} / {res['max']:>3} pts | {res['sim']:>5.1f}%    ｜|{bar}")
        else: output.append(f"{i+1}位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|{'□'*20}")
    
    output.extend(["-" * 80, " 【リアルタイム認識ログ（直近5件）】", "-" * 80])
    if detail_logs:
        recent_logs = detail_logs[-5:][::-1]
        for log in recent_logs:
            output.append(f" [{log[0]}] 単語: {log[1]:<8} ➔ {log[2]:>3}番札に加点 (スコア: {log[3]}/{log[4]}, 類似度: {log[5]})")
    else:
        output.append(" まだキーワードが認識されていません。")
        
    output.extend(["=" * 80])
    print("\n".join(output))

def save_to_result_csv(data):
    os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
    file_exists = os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f) 
        if not file_exists: writer.writerow(["日時", "ファイル名", "前の札", "対象札", "最終推定ID", "最終類似度", "認識単語"])
        writer.writerow(data)

# --- 4. メイン処理 ---

def run_realtime_karuta_with_record():
    print("【使用モデルを選択してください】")
    choice = input("1: Standard (精度重視) / 2: Small (速度重視) -> ")
    model_name = "model_standard" if choice == "1" else "model_small"
    prev_name = input("【前の札名を入力してください】: ")
    target_name = input("【識別したい札名を入力してください】: ")
    
    grammar, tag_map, max_scores = prepare_karuta_system(CSV_FILE)
    if not tag_map: return

    print(f"\n--- モデル読み込み中 ({model_name}) ---")
    model = Model(model_name)
    rec = KaldiRecognizer(model, SAMPLERATE, json.dumps(grammar, ensure_ascii=False))

    dt = datetime.now(); dt_str = dt.strftime("%Y%m%d_%H%M%S")
    base_name = f"{prev_name}_{target_name}_{dt_str}"
    save_filename = f"{base_name}.wav"
    detail_log_filename = f"{base_name}.csv" 
    
    os.makedirs("recordings", exist_ok=True)
    os.makedirs("results", exist_ok=True)

    full_path_wav = os.path.join("recordings", save_filename)
    full_path_csv = os.path.join("results", detail_log_filename) 

    q = queue.Queue(); recorded_frames = [] 
    def callback(indata, frames, time, status):
        data = bytes(indata); q.put(data); recorded_frames.append(data)

    scores = {}; seen_words_per_card = {}; all_detected_words = []
    detail_logs = []

    # ★音声ストリームに入る直前に、開始時刻を記録
    start_time = time.time()

    try:
        with sd.RawInputStream(samplerate=SAMPLERATE, blocksize=4000, 
                               channels=1, dtype='int16', callback=callback):
            while True:
                data = q.get()
                
                # ★現在の経過時間を計算
                elapsed = time.time() - start_time
                is_skipping = (elapsed < SKIP_SECONDS)

                if not rec.AcceptWaveform(data):
                    partial = json.loads(rec.PartialResult()).get('partial', "")
                    if partial:
                        new_words = partial.split()
                        for word in new_words:
                            # ★スキップ時間（6.0秒）を過ぎている場合のみ、加点処理を行う
                            if not is_skipping:
                                if word in tag_map:
                                    for tid in tag_map[word]:
                                        if tid not in seen_words_per_card: seen_words_per_card[tid] = set()
                                        if word not in seen_words_per_card[tid]:
                                            scores[tid] = scores.get(tid, 0) + len(word)
                                            seen_words_per_card[tid].add(word)
                                            if word not in all_detected_words: all_detected_words.append(word)
                                            
                                            current_sim = (scores[tid] / max_scores.get(tid, 1)) * 100
                                            detail_logs.append([
                                                datetime.now().strftime("%H:%M:%S.%f")[:-3],
                                                word, tid, scores[tid], max_scores.get(tid), f"{current_sim:.1f}%"
                                            ])
                    
                    # ダッシュボード表示に「is_skipping」と「elapsed」の状態を渡す
                    print_dashboard(scores, max_scores, partial, model_name, prev_name, target_name, detail_logs, is_skipping, elapsed)

    except KeyboardInterrupt:
        final_id, final_sim = "---", 0.0
        if scores:
            final_id = max(scores, key=lambda k: scores[k]/max_scores.get(k, 1))
            final_sim = (scores[final_id] / max_scores.get(final_id, 1)) * 100

        # WAV保存（recordings/配下）
        with wave.open(full_path_wav, 'wb') as wf:
            wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(SAMPLERATE)
            wf.writeframes(b''.join(recorded_frames))
        
        # 詳細ログCSV保存 (results/配下)
        with open(full_path_csv, "w", encoding="utf-8-sig", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["時刻", "認識単語", "加点札ID", "現在スコア", "満点スコア", "類似度"])
            writer.writerows(detail_logs)

        # 全体履歴CSV保存（results/配下）
        save_to_result_csv([dt.strftime("%Y-%m-%d %H:%M:%S"), save_filename, prev_name, target_name, final_id, f"{final_sim:.1f}%", " ".join(all_detected_words)])
        
        print(f"\n録音保存: {full_path_wav}")
        print(f"詳細ログ保存: {full_path_csv}")
        print(f"最終推定: {final_id} ({final_sim:.1f}%)")

if __name__ == "__main__":
    run_realtime_karuta_with_record()

設定: [Model: model_small] [Prev: nanishi] [Target: haruno]
状況: 【類似度判定中: 17.6s経過】(加点有効)
 | 認識中の言葉  | 春の ゆめ ばか 山桜                                       
--------------------------------------------------------------------------------
順位  ｜推定札        | 加点/満点       ｜類似度      ｜棒グラフ
--------------------------------------------------------------------------------
1位 ｜  67番      |   8 /  46 pts |  17.4%    ｜|■■■□□□□□□□□□□□□□□□□□
2位 ｜  33番      |   4 /  45 pts |   8.9%    ｜|■□□□□□□□□□□□□□□□□□□□
3位 ｜  15番      |   4 /  49 pts |   8.2%    ｜|■□□□□□□□□□□□□□□□□□□□
4位 ｜  21番      |   3 /  46 pts |   6.5%    ｜|■□□□□□□□□□□□□□□□□□□□
5位 ｜  12番      |   2 /  45 pts |   4.4%    ｜|□□□□□□□□□□□□□□□□□□□□
--------------------------------------------------------------------------------
 【リアルタイム認識ログ（直近5件）】
--------------------------------------------------------------------------------
 [15:53:42.595] 単語: 山桜       ➔  66番札に加点 (スコア: 2/45, 類似度: 4.4%)
 [15:53:42.075] 単語: あま       ➔  90番札に加点 (スコア: 2/46, 類似度: 4.3%)
 [15

辞書内の単語が使われたら音声検出ってしてる。ただこれは、下の句がCSVファイルに含まれていないので、うまくいかないときもある

In [12]:
import sounddevice as sd
import queue
import json
import sys
import os
import csv
import wave
import time  # 経過時間測定用
from datetime import datetime
from vosk import Model, KaldiRecognizer
from IPython.display import clear_output, display

# --- 1. 設定 ---
CSV_FILE = "data/karuta_data.csv"
RESULTS_CSV = "results/recognition_history.csv" 
SAMPLERATE = 16000

# 前の札の下の句を無視する時間（秒）
SKIP_SECONDS = 6.0

# --- 2. 辞書ロジック ---

def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    if pos_index == 0 or pos_index >= 2:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    return p

def extract_kanji_parts(text):
    text = text.replace(" ", "")
    p = []
    if len(text) >= 2: p.append(text[:2])
    if len(text) >= 3: p.append(text[:3])
    if len(text) >= 2: p.append(text[-2:])
    if len(text) >= 3: p.append(text[-3:])
    return p

def prepare_karuta_system(file_path):
    if not os.path.exists(file_path):
        print(f"Error: {file_path} が見つかりません。")
        return None, None, None
    raw_cards = []
    with open(file_path, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            words = []; seen = set()
            def add_words(new_words, is_kimari=False):
                for w in new_words:
                    if w in seen: continue
                    if len(w) >= (1 if is_kimari else 2):
                        words.append(w); seen.add(w)
            add_words([kimari], is_kimari=True)
            add_words([gendai.replace(" ", "")])
            k_blocks = kanji.split(); h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks): add_words(extract_kanji_parts(k_blocks[i]))
                if i < len(h_blocks): add_words(extract_kana_parts(h_blocks[i], i))
            raw_cards.append({"num": num, "words": words})
    
    if not raw_cards: return None, None, None
    min_count = min(len(c["words"]) for c in raw_cards)
    tag_map = {}; grammar_list = []; card_max_scores = {}
    for c in raw_cards:
        num = c["num"]
        unified_words = c["words"][:min_count]
        total_potential = 0
        for w in unified_words:
            grammar_list.append(w)
            tag_map.setdefault(w, []).append(num)
            total_potential += len(w)
        card_max_scores[num] = total_potential
    return list(set(grammar_list)), tag_map, card_max_scores

# --- 3. 表示と保存の関数 ---

# 引数「is_skipping」を状態を表す文字列「status_mode」に変更
def print_dashboard(scores, max_scores, partial_text, model_name, prev_札, target_札, detail_logs, status_mode, elapsed):
    clear_output(wait=True)
    results = []
    for tid, score in scores.items():
        max_s = max_scores.get(tid, 1)
        sim = (score / max_s) * 100
        results.append({"id": tid, "score": score, "max": max_s, "sim": sim})
    top_5 = sorted(results, key=lambda x: x['sim'], reverse=True)[:5]
    
    # 状況表示ラベルの作成
    if status_mode == "wait":
        status_label = "【音声検出待ち...】(声が入るとカウントを開始します)"
    elif status_mode == "skip":
        status_label = f"【下の句スキップ中: {elapsed:.1f}s / {SKIP_SECONDS:.1f}s】(加点無効)"
    else:
        status_label = f"【類似度判定中: {elapsed:.1f}s経過】(加点有効)"

    output = ["=" * 80, 
              f"設定: [Model: {model_name}] [Prev: {prev_札}] [Target: {target_札}]", 
              f"状況: {status_label}", "=" * 80]
              
    clean_partial = partial_text.replace("\n", " ")
    display_partial = clean_partial[-50:] if len(clean_partial) > 50 else clean_partial
    output.append(f" | 認識中の言葉  | {display_partial.ljust(50)}")
    output.extend(["-" * 80, f"{'順位':<4}｜{'推定札':<10} | {'加点/満点':<12}｜{'類似度':<8} ｜棒グラフ", "-" * 80])
    for i in range(5):
        if i < len(top_5):
            res = top_5[i]; bar_count = int(res['sim'] / 5); bar = "■" * bar_count + "□" * (20 - bar_count)
            output.append(f"{i+1}位 ｜ {res['id']:>3}番      | {res['score']:>3} / {res['max']:>3} pts | {res['sim']:>5.1f}%    ｜|{bar}")
        else: output.append(f"{i+1}位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|{'□'*20}")
    
    output.extend(["-" * 80, " 【リアルタイム認識ログ（直近5件）】", "-" * 80])
    if detail_logs:
        recent_logs = detail_logs[-5:][::-1]
        for log in recent_logs:
            output.append(f" [{log[0]}] 単語: {log[1]:<8} ➔ {log[2]:>3}番札に加点 (スコア: {log[3]}/{log[4]}, 類似度: {log[5]})")
    else:
        output.append(" まだキーワードが認識されていません。")
        
    output.extend(["=" * 80])
    print("\n".join(output))

def save_to_result_csv(data):
    os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
    file_exists = os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f) 
        if not file_exists: writer.writerow(["日時", "ファイル名", "前の札", "対象札", "最終推定ID", "最終類似度", "認識単語"])
        writer.writerow(data)

# --- 4. メイン処理 ---

def run_realtime_karuta_with_record():
    print("【使用モデルを選択してください】")
    choice = input("1: Standard (精度重視) / 2: Small (速度重視) -> ")
    model_name = "model_standard" if choice == "1" else "model_small"
    prev_name = input("【前の札名を入力してください】: ")
    target_name = input("【識別したい札名を入力してください】: ")
    
    grammar, tag_map, max_scores = prepare_karuta_system(CSV_FILE)
    if not tag_map: return

    print(f"\n--- モデル読み込み中 ({model_name}) ---")
    model = Model(model_name)
    rec = KaldiRecognizer(model, SAMPLERATE, json.dumps(grammar, ensure_ascii=False))

    dt = datetime.now(); dt_str = dt.strftime("%Y%m%d_%H%M%S")
    base_name = f"{prev_name}_{target_name}_{dt_str}"
    save_filename = f"{base_name}.wav"
    detail_log_filename = f"{base_name}.csv" 
    
    os.makedirs("recordings", exist_ok=True)
    os.makedirs("results", exist_ok=True)

    full_path_wav = os.path.join("recordings", save_filename)
    full_path_csv = os.path.join("results", detail_log_filename) 

    q = queue.Queue(); recorded_frames = [] 
    def callback(indata, frames, time, status):
        data = bytes(indata); q.put(data); recorded_frames.append(data)

    scores = {}; seen_words_per_card = {}; all_detected_words = []
    detail_logs = []

    # 音声検出のトリガー用変数
    audio_detected_time = None  # 音声を最初に検知した時刻

    try:
        with sd.RawInputStream(samplerate=SAMPLERATE, blocksize=4000, 
                               channels=1, dtype='int16', callback=callback):
            while True:
                data = q.get()
                
                # Voskにデータを送りつつ、現在の状態を取得
                is_recognized = rec.AcceptWaveform(data)
                partial = json.loads(rec.PartialResult()).get('partial', "")

                # ★【音声検出ロジック】部分認識（partial）に何か文字が入った瞬間をスタートとする
                if audio_detected_time is None and partial.strip() != "":
                    audio_detected_time = time.time()

                # 時間とモードの判定
                if audio_detected_time is None:
                    # まだ声が検出されていない状態
                    elapsed = 0.0
                    status_mode = "wait"
                    is_skipping = True
                else:
                    # 声を検出した後の経過時間
                    elapsed = time.time() - audio_detected_time
                    if elapsed < SKIP_SECONDS:
                        status_mode = "skip"
                        is_skipping = True
                    else:
                        status_mode = "run"
                        is_skipping = False

                if not is_recognized:
                    if partial:
                        new_words = partial.split()
                        for word in new_words:
                            # スキップ時間（6秒間）が終わるまでは加点しない
                            if not is_skipping:
                                if word in tag_map:
                                    for tid in tag_map[word]:
                                        if tid not in seen_words_per_card: seen_words_per_card[tid] = set()
                                        if word not in seen_words_per_card[tid]:
                                            scores[tid] = scores.get(tid, 0) + len(word)
                                            seen_words_per_card[tid].add(word)
                                            if word not in all_detected_words: all_detected_words.append(word)
                                            
                                            current_sim = (scores[tid] / max_scores.get(tid, 1)) * 100
                                            detail_logs.append([
                                                datetime.now().strftime("%H:%M:%S.%f")[:-3],
                                                word, tid, scores[tid], max_scores.get(tid), f"{current_sim:.1f}%"
                                            ])
                    
                    # ダッシュボード表示を更新
                    print_dashboard(scores, max_scores, partial, model_name, prev_name, target_name, detail_logs, status_mode, elapsed)

    except KeyboardInterrupt:
        final_id, final_sim = "---", 0.0
        if scores:
            final_id = max(scores, key=lambda k: scores[k]/max_scores.get(k, 1))
            final_sim = (scores[final_id] / max_scores.get(final_id, 1)) * 100

        # WAV保存（recordings/配下）
        with wave.open(full_path_wav, 'wb') as wf:
            wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(SAMPLERATE)
            wf.writeframes(b''.join(recorded_frames))
        
        # 詳細ログCSV保存 (results/配下)
        with open(full_path_csv, "w", encoding="utf-8-sig", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["時刻", "認識単語", "加点札ID", "現在スコア", "満点スコア", "類似度"])
            writer.writerows(detail_logs)

        # 全体履歴CSV保存（results/配下）
        save_to_result_csv([dt.strftime("%Y-%m-%d %H:%M:%S"), save_filename, prev_name, target_name, final_id, f"{final_sim:.1f}%", " ".join(all_detected_words)])
        
        print(f"\n録音保存: {full_path_wav}")
        print(f"詳細ログ保存: {full_path_csv}")
        print(f"最終推定: {final_id} ({final_sim:.1f}%)")

if __name__ == "__main__":
    run_realtime_karuta_with_record()

設定: [Model: model_small] [Prev: haruno] [Target: harusu]
状況: 【下の句スキップ中: 4.3s / 6.0s】(加点無効)
 | 認識中の言葉  | すぎ 月見 白妙                                          
--------------------------------------------------------------------------------
順位  ｜推定札        | 加点/満点       ｜類似度      ｜棒グラフ
--------------------------------------------------------------------------------
1位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|□□□□□□□□□□□□□□□□□□□□
2位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|□□□□□□□□□□□□□□□□□□□□
3位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|□□□□□□□□□□□□□□□□□□□□
4位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|□□□□□□□□□□□□□□□□□□□□
5位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|□□□□□□□□□□□□□□□□□□□□
--------------------------------------------------------------------------------
 【リアルタイム認識ログ（直近5件）】
--------------------------------------------------------------------------------
 まだキーワードが認識されていません。

録音保存: recordings\haruno_harusu_20260517_155607.wav
詳細ログ保存: results\haruno_harusu_20260517_155607.csv
最終推

こっちはがちの音声検出

In [ ]:
import sounddevice as sd
import queue
import json
import sys
import os
import csv
import wave
import time
import numpy as np
import torch  # ★Silero VAD用に追記
from datetime import datetime
from vosk import Model, KaldiRecognizer
from IPython.display import clear_output, display

# --- 1. 設定 ---
CSV_FILE = "data/karuta_data.csv"
RESULTS_CSV = "results/recognition_history.csv" 
SAMPLERATE = 16000

# 前の札の下の句を無視する時間（秒）
SKIP_SECONDS = 6.0

# ★【変更】VADの判定しきい値（0.0 〜 1.0）。0.5前後が標準で、値を上げるとノイズに強くなります
VAD_THRESHOLD = 0.5

# --- 2. 辞書ロジック ---

def extract_kana_parts(text, pos_index):
    text = text.replace(" ", "")
    p = []
    L = len(text)
    if L < 2: return [text]
    if pos_index == 0 or pos_index >= 2:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 3: p.append(text[-3:])
        p.append(text[-2:])
    elif pos_index == 1:
        p.append(text[0:2])
        if L >= 4: p.append(text[2:4])
        if L >= 6: p.append(text[4:6])
        if L >= 7: p.append(text[-3:])
        if L >= 4: p.append(text[1:4])
        p.append(text[-2:])
    return p

def extract_kanji_parts(text):
    text = text.replace(" ", "")
    p = []
    if len(text) >= 2: p.append(text[:2])
    if len(text) >= 3: p.append(text[:3])
    if len(text) >= 2: p.append(text[-2:])
    if len(text) >= 3: p.append(text[-3:])
    return p

def prepare_karuta_system(file_path):
    if not os.path.exists(file_path):
        print(f"Error: {file_path} が見つかりません。")
        return None, None, None
    raw_cards = []
    with open(file_path, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5: continue
            num, kanji, kana, kimari, gendai = [r.strip() for r in row[:5]]
            words = []; seen = set()
            def add_words(new_words, is_kimari=False):
                for w in new_words:
                    if w in seen: continue
                    if len(w) >= (1 if is_kimari else 2):
                        words.append(w); seen.add(w)
            add_words([kimari], is_kimari=True)
            add_words([gendai.replace(" ", "")])
            k_blocks = kanji.split(); h_blocks = kana.split()
            for i in range(max(len(k_blocks), len(h_blocks))):
                if i < len(k_blocks): add_words(extract_kanji_parts(k_blocks[i]))
                if i < len(h_blocks): add_words(extract_kana_parts(h_blocks[i], i))
            raw_cards.append({"num": num, "words": words})
    
    if not raw_cards: return None, None, None
    
    tag_map = {}; grammar_list = []; card_max_scores = {}
    for c in raw_cards:
        num = c["num"]
        unified_words = c["words"]
        total_potential = 0
        for w in unified_words:
            grammar_list.append(w)
            tag_map.setdefault(w, []).append(num)
            total_potential += len(w)
        card_max_scores[num] = total_potential
        
    return list(set(grammar_list)), tag_map, card_max_scores

# --- 3. 表示と保存の関数 ---

def print_dashboard(scores, max_scores, partial_text, model_name, prev_札, target_札, detail_logs, status_mode, elapsed, speech_prob):
    clear_output(wait=True)
    results = []
    for tid, score in scores.items():
        max_s = max_scores.get(tid, 1)
        sim = (score / max_s) * 100
        results.append({"id": tid, "score": score, "max": max_s, "sim": sim})
    top_5 = sorted(results, key=lambda x: x['sim'], reverse=True)[:5]
    
    # ★ 状態表示をVAD仕様に変更
    if status_mode == "wait":
        status_label = f"【音声検出待ち...】(VAD音声確率: {speech_prob*100:.1f}% / しきい値: {VAD_THRESHOLD*100:.1f}%)"
    elif status_mode == "skip":
        status_label = f"【下の句スキップ中: {elapsed:.1f}s / {SKIP_SECONDS:.1f}s】(加点無効)"
    else:
        status_label = f"【類似度判定中: {elapsed:.1f}s経過】(加点有効)"

    output = ["=" * 80, 
              f"設定: [Model: {model_name}] [Prev: {prev_札}] [Target: {target_札}]", 
              f"状況: {status_label}", "=" * 80]
              
    clean_partial = partial_text.replace("\n", " ")
    display_partial = clean_partial[-50:] if len(clean_partial) > 50 else clean_partial
    output.append(f" | 認識中の言葉  | {display_partial.ljust(50)}")
    output.extend(["-" * 80, f"{'順位':<4}｜{'推定札':<10} | {'加点/満点':<12}｜{'類似度':<8} ｜棒グラフ", "-" * 80])
    for i in range(5):
        if i < len(top_5):
            res = top_5[i]; bar_count = int(res['sim'] / 5); bar = "■" * bar_count + "□" * (20 - bar_count)
            output.append(f"{i+1}位 ｜ {res['id']:>3}番      | {res['score']:>3} / {res['max']:>3} pts | {res['sim']:>5.1f}%    ｜|{bar}")
        else: output.append(f"{i+1}位 ｜ ---番      |   0 / --- pts |   0.0%    ｜|{'□'*20}")
    
    output.extend(["-" * 80, " 【リアルタイム認識ログ（直近5件）】", "-" * 80])
    if detail_logs:
        recent_logs = detail_logs[-5:][::-1]
        for log in recent_logs:
            output.append(f" [{log[0]}] 単語: {log[1]:<8} ➔ {log[2]:>3}番札に加点 (スコア: {log[3]}/{log[4]}, 類似度: {log[5]})")
    else:
        output.append(" まだキーワードが認識されていません。")
        
    output.extend(["=" * 80])
    print("\n".join(output))

def save_to_result_csv(data):
    os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
    file_exists = os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f) 
        if not file_exists: writer.writerow(["日時", "ファイル名", "前の札", "対象札", "最終推定ID", "最終類似度", "認識単語"])
        writer.writerow(data)

# --- 4. メイン処理 ---

def run_realtime_karuta_with_record():
    print("【使用モデルを選択してください】")
    choice = input("1: Standard (精度重視) / 2: Small (速度重視) -> ")
    model_name = "model_standard" if choice == "1" else "model_small"
    prev_name = input("【前の札名を入力してください】: ")
    target_name = input("【識別したい札名を入力してください】: ")
    
    grammar, tag_map, max_scores = prepare_karuta_system(CSV_FILE)
    if not tag_map: return

    # ★【新規】Silero VAD モデルの読み込み（ローカルに自動キャッシュされます）
    print("\n--- Silero VAD 読み込み中 ---")
    vad_model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                                      model='silero_vad',
                                      force_reload=False,
                                      trust_repo=True)
    (get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

    print(f"\n--- Vosk モデル読み込み中 ({model_name}) ---")
    model = Model(model_name)
    rec = KaldiRecognizer(model, SAMPLERATE, json.dumps(grammar, ensure_ascii=False))

    dt = datetime.now(); dt_str = dt.strftime("%Y%m%d_%H%M%S")
    base_name = f"{prev_name}_{target_name}_{dt_str}"
    save_filename = f"{base_name}.wav"
    detail_log_filename = f"{base_name}.csv" 
    
    os.makedirs("recordings", exist_ok=True)
    os.makedirs("results", exist_ok=True)

    full_path_wav = os.path.join("recordings", save_filename)
    full_path_csv = os.path.join("results", detail_log_filename) 

    q = queue.Queue(); recorded_frames = [] 
    def callback(indata, frames, time, status):
        data = bytes(indata); q.put(data); recorded_frames.append(data)

    scores = {}; seen_words_per_card = {}; all_detected_words = []
    detail_logs = []
    
    # トリガー制御用の変数
    audio_detected_time = None 
    speech_prob = 0.0

    try:
        # ★ blocksizeを Silero VAD が受け付ける 512 に変更
        with sd.RawInputStream(samplerate=SAMPLERATE, blocksize=512, 
                               channels=1, dtype='int16', callback=callback):
            while True:
                data = q.get()
                
                # --- ★ Silero VAD による音声解析 ---
                # 16bit int のバイトデータを、VADが処理できる float32 テンソルに変換
                audio_data = np.frombuffer(data, dtype=np.int16).astype(np.float32) / 32768.0
                audio_tensor = torch.from_numpy(audio_data)
                
                # 現在のチャンクが「話し声」である確率を計算
                speech_prob = vad_model(audio_tensor, SAMPLERATE).item()

                # Voskにデータを供給
                if not rec.AcceptWaveform(data):
                    result_text = json.loads(rec.PartialResult()).get('partial', "")
                else:
                    result_text = json.loads(rec.Result()).get('text', "")

                # --- ★ 状態管理とスキップロジック ---
                # 最初の一回、音声確率がしきい値を超えたらタイマーをスタート（発話検知フラグ）
                if audio_detected_time is None and speech_prob > VAD_THRESHOLD:
                    audio_detected_time = time.time()

                if audio_detected_time is None:
                    elapsed = 0.0
                    status_mode = "wait"
                    is_skipping = True
                else:
                    elapsed = time.time() - audio_detected_time
                    if elapsed < SKIP_SECONDS:
                        status_mode = "skip"
                        is_skipping = True
                    else:
                        status_mode = "run"
                        is_skipping = False

                # --- 加点処理 ---
                if result_text:
                    new_words = result_text.split()
                    for word in new_words:
                        if not is_skipping:
                            if word in tag_map:
                                for tid in tag_map[word]:
                                    if tid not in seen_words_per_card: 
                                        seen_words_per_card[tid] = set()
                                    if word not in seen_words_per_card[tid]:
                                        scores[tid] = scores.get(tid, 0) + len(word)
                                        seen_words_per_card[tid].add(word)
                                        if word not in all_detected_words: 
                                            all_detected_words.append(word)
                                        
                                        current_sim = (scores[tid] / max_scores.get(tid, 1)) * 100
                                        detail_logs.append([
                                            datetime.now().strftime("%H:%M:%S.%f")[:-3],
                                            word, tid, scores[tid], max_scores.get(tid), f"{current_sim:.1f}%"
                                        ])
                
                # ダッシュボードへの反映
                print_dashboard(scores, max_scores, result_text, model_name, prev_name, target_name, detail_logs, status_mode, elapsed, speech_prob)

    except KeyboardInterrupt:
        final_id, final_sim = "---", 0.0
        if scores:
            final_id = max(scores, key=lambda k: scores[k]/max_scores.get(k, 1))
            final_sim = (scores[final_id] / max_scores.get(final_id, 1)) * 100

        # WAV保存
        with wave.open(full_path_wav, 'wb') as wf:
            wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(SAMPLERATE)
            wf.writeframes(b''.join(recorded_frames))
        
        # 詳細ログCSV保存
        with open(full_path_csv, "w", encoding="utf-8-sig", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["時刻", "認識単語", "加点札ID", "現在スコア", "満点スコア", "類似度"])
            writer.writerows(detail_logs)

        # 全体履歴CSV保存
        save_to_result_csv([dt.strftime("%Y-%m-%d %H:%M:%S"), save_filename, prev_name, target_name, final_id, f"{final_sim:.1f}%", " ".join(all_detected_words)])
        
        print(f"\n録音保存: {full_path_wav}")
        print(f"詳細ログ保存: {full_path_csv}")
        print(f"最終推定: {final_id} ({final_sim:.1f}%)")

if __name__ == "__main__":
    run_realtime_karuta_with_record()